In [1]:
import pandas as pd

# --- 1. Load the data --
df = pd.read_csv("../data/features/features_core_v2.csv")


In [2]:
# --- 2. Parse the timestamp column as a real datetime ---
df["timestamp_hour"] = pd.to_datetime(df["timestamp_hour"])

In [3]:
# --- 3. Basic shape check against what the guide promises ---
print("Shape (rows, columns):", df.shape)
print("Expected: (78204, 145)")

print("\nNumber of unique stations:", df["station_name"].nunique())
print("Expected: 38")

print("\nDate range:")
print("  Start:", df["timestamp_hour"].min())
print("  End:  ", df["timestamp_hour"].max())

Shape (rows, columns): (78204, 145)
Expected: (78204, 145)

Number of unique stations: 38
Expected: 38

Date range:
  Start: 2026-04-10 21:00:00
  End:   2026-07-09 18:00:00


In [4]:
# --- 4. Check for duplicate station-hour rows ---
# A "station-hour" is one station at one timestamp. There should be
# exactly one row per station per hour — never two.
duplicate_count = df.duplicated(subset=["station_name", "timestamp_hour"]).sum()
print("\nDuplicate station-hour rows:", duplicate_count)
print("Expected: 0")


Duplicate station-hour rows: 0
Expected: 0


In [5]:
# --- 5. Identify target columns vs feature columns ---
target_columns = [c for c in df.columns if c.startswith("target_aqi_")]
feature_columns = [c for c in df.columns if c not in target_columns]

print("\nNumber of target columns found:", len(target_columns))
print("Expected: 24")
print("Number of feature columns:", len(feature_columns))


Number of target columns found: 24
Expected: 24
Number of feature columns: 121


In [6]:
# --- 6. Peek at the target column names to confirm they look right ---
print("\nTarget columns:", target_columns)


Target columns: ['target_aqi_1h', 'target_aqi_2h', 'target_aqi_3h', 'target_aqi_4h', 'target_aqi_5h', 'target_aqi_6h', 'target_aqi_7h', 'target_aqi_8h', 'target_aqi_9h', 'target_aqi_10h', 'target_aqi_11h', 'target_aqi_12h', 'target_aqi_13h', 'target_aqi_14h', 'target_aqi_15h', 'target_aqi_16h', 'target_aqi_17h', 'target_aqi_18h', 'target_aqi_19h', 'target_aqi_20h', 'target_aqi_21h', 'target_aqi_22h', 'target_aqi_23h', 'target_aqi_24h']


In [7]:
# --- 7. Group columns into mutually exclusive families ---
identifier_cols = [c for c in ["location_id", "station_name", "provider_name", "timestamp_hour", "latitude", "longitude"] if c in df.columns]
pollutant_observed_cols = [c for c in df.columns if c.endswith("_observed")]
live_context_cols = [c for c in ["current_aqi", "aqi_calculation_valid", "temperature_2m", "relative_humidity_2m", "precipitation", "surface_pressure", "is_raining", "wind_speed_10m_ms", "wind_u_ms", "wind_v_ms"] if c in df.columns]
current_measurement_cols = pollutant_observed_cols + live_context_cols
quality_flag_cols = [c for c in df.columns if c.startswith(("has_", "station_seen_")) or c.endswith(("_temporary_missing_causal", "_not_yet_observed_causal"))]
target_columns = [c for c in df.columns if c.startswith("target_aqi_")]
already_grouped = set(identifier_cols + current_measurement_cols + quality_flag_cols + target_columns)
engineered_cols = [c for c in df.columns if c not in already_grouped]
family_counts = {
    "Identifiers": len(identifier_cols),
    "Current measurements": len(current_measurement_cols),
    "Quality/history flags": len(quality_flag_cols),
    "Targets": len(target_columns),
    "Engineered predictors": len(engineered_cols),
}
for label, count in family_counts.items():
    print(f"{label}: {count}")
print("Total accounted for:", sum(family_counts.values()))
print("Should equal total columns:", len(df.columns))
assert sum(family_counts.values()) == len(df.columns), "Column families overlap or omit columns"

Identifiers: 6
Current measurements: 16
Quality/history flags: 24
Targets: 24
Engineered predictors: 75
Total accounted for: 145
Should equal total columns: 145


In [8]:
preview_cols = [c for c in ["pm25_observed", "pm10_observed", "no2_observed", "co_observed", "so2_observed", "o3_observed", "current_aqi"] if c in df]
df[preview_cols].describe(include="all")

,pm25_observed,pm10_observed,no2_observed,co_observed,so2_observed,o3_observed,current_aqi
count,62828.000000,62334.000000,62919.000000,61786.000000,50450.000000,61704.000000,60828.000000
mean,55.178141,177.533931,34.969036,1.006543,17.190427,32.903660,161.513645
std,41.328977,111.649833,26.203716,0.682388,15.814872,28.095383,60.765535
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,29.000000
25%,30.300000,104.000000,16.600000,0.593000,9.000000,12.700000,119.000000
50%,45.000000,153.000000,28.000000,0.868000,14.000000,25.500000,151.000000
75%,67.300000,225.000000,45.100000,1.250000,22.200000,45.800000,189.000000
max,977.000000,1480.000000,637.000000,9.730000,1170.000000,578.000000,478.000000


In [9]:
# --- 8. Check quality flags against documented proxy-imputation rates ---
for col in quality_flag_cols:
    pct = 100 * df[col].sum() / len(df)
    print(f"{col:30s} -> {df[col].sum():>6} rows flagged ({pct:.1f}%)")

has_pm25                       ->  62828 rows flagged (80.3%)
has_pm10                       ->  62334 rows flagged (79.7%)
has_no2                        ->  62919 rows flagged (80.5%)
has_co                         ->  61786 rows flagged (79.0%)
has_so2                        ->  50450 rows flagged (64.5%)
has_o3                         ->  61704 rows flagged (78.9%)
station_seen_pm25_so_far       ->  76463 rows flagged (97.8%)
station_seen_pm10_so_far       ->  74387 rows flagged (95.1%)
station_seen_no2_so_far        ->  76466 rows flagged (97.8%)
station_seen_co_so_far         ->  76466 rows flagged (97.8%)
station_seen_so2_so_far        ->  62053 rows flagged (79.3%)
station_seen_o3_so_far         ->  74408 rows flagged (95.1%)
pm25_temporary_missing_causal  ->  13635 rows flagged (17.4%)
pm10_temporary_missing_causal  ->  12053 rows flagged (15.4%)
no2_temporary_missing_causal   ->  13547 rows flagged (17.3%)
co_temporary_missing_causal    ->  14680 rows flagged (18.8%)
so2_temp

In [10]:
# --- 9. Understand target missingness ---
target_missing = df[target_columns].isna().sum()
print(target_missing)

target_aqi_1h     17376
target_aqi_2h     17376
target_aqi_3h     17376
target_aqi_4h     17376
target_aqi_5h     17376
target_aqi_6h     17376
target_aqi_7h     17376
target_aqi_8h     17376
target_aqi_9h     17376
target_aqi_10h    17376
target_aqi_11h    17376
target_aqi_12h    17376
target_aqi_13h    17376
target_aqi_14h    17376
target_aqi_15h    17376
target_aqi_16h    17376
target_aqi_17h    17376
target_aqi_18h    17386
target_aqi_19h    17437
target_aqi_20h    17552
target_aqi_21h    17683
target_aqi_22h    17819
target_aqi_23h    17964
target_aqi_24h    18115
dtype: int64


In [11]:
# --- 10. Quick look at station coverage ---
station_counts = df["station_name"].value_counts()
print("Number of stations:", len(station_counts))
print(station_counts.describe())

Number of stations: 38
count      38.0
mean     2058.0
std         0.0
min      2058.0
25%      2058.0
50%      2058.0
75%      2058.0
max      2058.0
Name: count, dtype: float64


In [12]:
# Step 3a: Chronological split boundaries (same for every station)
# We use the full range of unique timestamps, sorted in time order.

unique_timestamps = sorted(df["timestamp_hour"].unique())
n = len(unique_timestamps)

# 70% of the timeline for training, next 15% for validation, last 15% for test.
train_end_idx = int(n * 0.70)
val_end_idx = int(n * 0.85)

train_end = unique_timestamps[train_end_idx]
val_end = unique_timestamps[val_end_idx]

print("Train:      up to", train_end)
print("Validation: ", train_end, "to", val_end)
print("Test:       ", val_end, "onward")

Train:      up to 2026-06-11 13:00:00
Validation:  2026-06-11 13:00:00 to 2026-06-26 22:00:00
Test:        2026-06-26 22:00:00 onward


In [13]:
# Step 3b: Build purged train/validation/test dataframes.
# Keep every row with the horizon target; only the persistence comparison later
# is restricted to rows where current AQI is also available.

def purged_split_for_horizon(df, horizon_hours, train_end, val_end):
    target_col = f"target_aqi_{horizon_hours}h"
    
    # For a row starting at time t, its true answer lives at t + horizon.
    label_time = df["timestamp_hour"] + pd.Timedelta(hours=horizon_hours)
    
    # TRAIN: keep only if BOTH the origin and the label are before train_end.
    is_train = (df["timestamp_hour"] < train_end) & (label_time < train_end)
    
    # VALIDATION: keep only if BOTH origin and label fall strictly inside 
    # the validation window (train_end to val_end).
    is_val = (
        (df["timestamp_hour"] >= train_end) & (df["timestamp_hour"] < val_end) &
        (label_time >= train_end) & (label_time < val_end)
    )
    
    # TEST: keep only if BOTH origin and label are at or after val_end.
    is_test = (
        (df["timestamp_hour"] >= val_end) &
        (label_time >= val_end)
    )
    
    train_df = df[is_train].dropna(subset=[target_col])
    val_df = df[is_val].dropna(subset=[target_col])
    test_df = df[is_test].dropna(subset=[target_col])
    
    return train_df, val_df, test_df

In [14]:
# Step 3c: Sanity check on the 6-hour horizon
train_6h, val_6h, test_6h = purged_split_for_horizon(
    df, horizon_hours=6, train_end=train_end, val_end=val_end
)

print("Train rows:", len(train_6h))
print("Val rows:  ", len(val_6h))
print("Test rows: ", len(test_6h))

print("\nTrain time range:", train_6h["timestamp_hour"].min(), "to", train_6h["timestamp_hour"].max())
print("Val time range:  ", val_6h["timestamp_hour"].min(), "to", val_6h["timestamp_hour"].max())
print("Test time range: ", test_6h["timestamp_hour"].min(), "to", test_6h["timestamp_hour"].max())

Train rows: 42203
Val rows:   8380
Test rows:  9860

Train time range: 2026-04-11 10:00:00 to 2026-06-11 06:00:00
Val time range:   2026-06-11 13:00:00 to 2026-06-26 15:00:00
Test time range:  2026-06-26 22:00:00 to 2026-07-09 12:00:00


In [15]:
# Step 3d: Prove the purge actually removed rows near the boundary

# Look at the last 8 hours before train_end, before any purging
near_boundary_raw = df[
    (df["timestamp_hour"] >= train_end - pd.Timedelta(hours=8)) &
    (df["timestamp_hour"] < train_end)
]

# Now check how many of those same rows survived into train_6h
near_boundary_survived = train_6h[
    (train_6h["timestamp_hour"] >= train_end - pd.Timedelta(hours=8)) &
    (train_6h["timestamp_hour"] < train_end)
]

print("Rows in the 8 hours before train_end (raw):", len(near_boundary_raw))
print("Of those, how many survived purging:       ", len(near_boundary_survived))

Rows in the 8 hours before train_end (raw): 304
Of those, how many survived purging:        43


In [16]:
# Step 3e: a purged train/val/test set for every one of the 24 forecast horizons.

splits_by_horizon = {}

for h in range(1, 25):
    train_df, val_df, test_df = purged_split_for_horizon(df, horizon_hours=h, train_end=train_end, val_end=val_end)
    splits_by_horizon[h] = {
        "train": train_df,
        "val": val_df,
        "test": test_df
    }
    print(f"Horizon {h:>2}h -> train: {len(train_df):>6}, val: {len(val_df):>6}, test: {len(test_df):>6}")

Horizon  1h -> train:  42203, val:   8534, test:  10032
Horizon  2h -> train:  42203, val:   8507, test:   9998
Horizon  3h -> train:  42203, val:   8478, test:   9964
Horizon  4h -> train:  42203, val:   8446, test:   9930
Horizon  5h -> train:  42203, val:   8413, test:   9895
Horizon  6h -> train:  42203, val:   8380, test:   9860
Horizon  7h -> train:  42203, val:   8346, test:   9825
Horizon  8h -> train:  42203, val:   8312, test:   9789
Horizon  9h -> train:  42203, val:   8278, test:   9753
Horizon 10h -> train:  42203, val:   8245, test:   9717
Horizon 11h -> train:  42203, val:   8212, test:   9681
Horizon 12h -> train:  42203, val:   8179, test:   9645
Horizon 13h -> train:  42203, val:   8146, test:   9609
Horizon 14h -> train:  42203, val:   8113, test:   9573
Horizon 15h -> train:  42203, val:   8082, test:   9537
Horizon 16h -> train:  42203, val:   8051, test:   9501
Horizon 17h -> train:  42203, val:   8020, test:   9465
Horizon 18h -> train:  42193, val:   7989, test:

In [17]:
import json

split_boundaries = {
    "train_end": str(train_end),
    "val_end": str(val_end),
    "full_range_start": str(df["timestamp_hour"].min()),
    "full_range_end": str(df["timestamp_hour"].max())
}

with open("../reports/split_boundaries.json", "w") as f:
    json.dump(split_boundaries, f, indent=2)

print(split_boundaries)

{'train_end': '2026-06-11 13:00:00', 'val_end': '2026-06-26 22:00:00', 'full_range_start': '2026-04-10 21:00:00', 'full_range_end': '2026-07-09 18:00:00'}


In [18]:
# Step 4a: Build the final, safe feature list

# Start from all columns, then explicitly remove anything unsafe or unusable.

# 1. Never allow any target column
unsafe_cols = list(target_columns)

# 2. Identifiers that shouldn't be raw numeric predictors (station_name will be encoded separately in Step 5 — not used raw here; lat/lon are kept since they're legitimate numeric spatial signal)
unsafe_cols += ["location_id", "provider_name", "timestamp_hour"]

# station_name is retained and encoded inside the training-only pipeline.

final_feature_columns = [c for c in df.columns if c not in unsafe_cols and c not in target_columns]

print("Final feature count:", len(final_feature_columns))
print(final_feature_columns)

Final feature count: 118
['station_name', 'latitude', 'longitude', 'current_aqi', 'aqi_calculation_valid', 'aqi_lag_1h', 'aqi_lag_6h', 'aqi_lag_12h', 'aqi_lag_24h', 'hour', 'day_of_week', 'is_weekend', 'month', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'surface_pressure', 'is_raining', 'wind_speed_10m_ms', 'wind_u_ms', 'wind_v_ms', 'pm25_observed', 'pm10_observed', 'no2_observed', 'co_observed', 'so2_observed', 'o3_observed', 'has_pm25', 'has_pm10', 'has_no2', 'has_co', 'has_so2', 'has_o3', 'station_seen_pm25_so_far', 'station_seen_pm10_so_far', 'station_seen_no2_so_far', 'station_seen_co_so_far', 'station_seen_so2_so_far', 'station_seen_o3_so_far', 'pm25_temporary_missing_causal', 'pm10_temporary_missing_causal', 'no2_temporary_missing_causal', 'co_temporary_missing_causal', 'so2_temporary_missing_causal', 'o3_temporary_missing_causal', 'pm25_not_yet_observed_causal', 'pm10_not_yet_observed_causal', 'no2_not_yet_observed_c

In [19]:
# Step 4b: Check missingness in the training partition only

# Use horizon 6's train split from Step 3 as our example
train_df = splits_by_horizon[6]["train"]

missing_pct = train_df[final_feature_columns].isna().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 0])

so2_observed_lag_6h               31.895837
so2_observed_lag_1h               31.592541
so2_observed                      31.459849
so2_observed_rolling_mean_6h      21.785181
co_observed_lag_24h               19.880103
o3_observed_lag_24h               19.692913
pm10_observed_lag_24h             18.692984
no2_observed_lag_24h              18.029524
aqi_lag_24h                       17.932374
pm25_observed_lag_24h             17.140962
co_observed_lag_6h                15.714523
o3_observed_lag_6h                15.397010
co_observed_lag_1h                15.269057
o3_observed_lag_1h                15.164799
co_observed                       15.117409
o3_observed                       15.072388
pm10_observed_lag_12h             14.404189
aqi_lag_12h                       14.359169
pm10_observed_lag_6h              14.340213
pm10_observed_lag_1h              13.991896
pm10_observed                     13.887638
no2_observed_lag_6h               13.662536
no2_observed_lag_1h             

In [20]:
# Step 4c: Decide: which columns can rely on the model to handle NaN natively, vs. which need explicit imputation

cols_with_missingness = missing_pct[missing_pct > 0].index.tolist()
print(f"{len(cols_with_missingness)} feature columns contain missing values.")
print("These numeric values are passed as-is; CatBoost handles numeric NaN values natively.")

55 feature columns contain missing values.
These numeric values are passed as-is; CatBoost handles numeric NaN values natively.


In [21]:
# Step 4d: Detect categorical predictors
categorical_cols = df[final_feature_columns].select_dtypes(include=["object", "category"]).columns.tolist()
print("Categorical predictors:", categorical_cols)

# Step 4e: Save the finalized feature list
with open("../reports/feature_list.txt", "w") as f:
    for col in final_feature_columns:
        f.write(col + "\n")

print("Saved", len(final_feature_columns), "features to reports/feature_list.txt")

Categorical predictors: ['station_name']
Saved 118 features to reports/feature_list.txt


In [22]:
# Step 5a: Build the pipeline

from catboost import CatBoostRegressor

# CatBoost receives station_name directly as a native categorical feature.

# Everything else in final_feature_columns is already numeric and can pass through untouched
numeric_cols = [c for c in final_feature_columns if c not in categorical_cols]

# We add station_name and dominant_pollutant BACK in here, specifically for this pipeline step
pipeline_input_cols = numeric_cols + categorical_cols

model_template = CatBoostRegressor(loss_function="MAE", eval_metric="MAE", iterations=1000, learning_rate=0.05, depth=8, l2_leaf_reg=5.0, random_seed=42, verbose=False, allow_writing_files=False)

print("CatBoost template built successfully.")
print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", categorical_cols)

CatBoost template built successfully.
Numeric columns: 117
Categorical columns: ['station_name']


In [23]:
#Test the pipeline structure on horizon 6

# Reuse the horizon-6 split from Step 3
train_df = splits_by_horizon[6]["train"]

X_train_sample = train_df[pipeline_input_cols].head(5)

print(X_train_sample[categorical_cols])
print("\nShape of input slice:", X_train_sample.shape)
print("Expected columns:", len(pipeline_input_cols))

               station_name
14  R K Puram, Delhi - DPCC
15  R K Puram, Delhi - DPCC
16  R K Puram, Delhi - DPCC
17  R K Puram, Delhi - DPCC
18  R K Puram, Delhi - DPCC

Shape of input slice: (5, 118)
Expected columns: 118


In [24]:
# Step 6a: Build and evaluate the persistence baseline for all 24 horizons

from sklearn.metrics import mean_absolute_error

persistence_results = []

for h in range(1, 25):
    target_col = f"target_aqi_{h}h"
    
    # Reuse this horizon's already-purged test set from Step 3
    test_df = splits_by_horizon[h]["test"]
    
    # Persistence baseline: current AQI is the prediction at h hours ahead.
    # test_df already contains only rows with both current and target AQI.
    eval_rows = test_df.dropna(subset=["current_aqi", target_col])
    
    y_true = eval_rows[target_col]
    y_pred_persistence = eval_rows["current_aqi"]
    
    mae = mean_absolute_error(y_true, y_pred_persistence)
    
    persistence_results.append({
        "horizon": h,
        "mae": mae,
        "n_rows_evaluated": len(eval_rows)
    })

persistence_df = pd.DataFrame(persistence_results)
print(persistence_df)

# Save this as a required deliverable
persistence_df.to_csv("../reports/persistence_baseline_metrics.csv", index=False)
print("Saved persistence baseline metrics.")

    horizon        mae  n_rows_evaluated
0         1   2.322516              9953
1         2   4.313096              9850
2         3   6.145639              9757
3         4   7.822307              9674
4         5   9.366934              9593
5         6  10.833018              9522
6         7  12.180982              9454
7         8  13.439876              9389
8         9  14.620161              9325
9        10  15.757775              9260
10       11  16.867493              9192
11       12  17.976104              9123
12       13  19.053899              9054
13       14  20.109146              8988
14       15  21.135111              8926
15       16  22.140891              8865
16       17  23.120005              8808
17       18  24.061728              8748
18       19  24.943045              8691
19       20  25.780584              8632
20       21  26.579923              8577
21       22  27.337675              8523
22       23  28.036823              8473
23       24  28.